In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import joblib
import os

from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported")

✅ Libraries imported


In [ ]:

# Load datasets
customers_df = pd.read_csv('../data/raw/customers.csv')
merchants_df = pd.read_csv('../data/raw/merchants.csv')
transactions_df = pd.read_csv('../data/raw/transactions.csv')

print(f"Customers: {customers_df.shape}")
print(f"Merchants: {merchants_df.shape}")
print(f"Transactions: {transactions_df.shape}")
print("\n✅ Data loaded")

Customers: (2000, 12)
Merchants: (500, 8)
Transactions: (10000, 14)

✅ Data loaded


In [ ]:
# Merge Datasets
# Merge transactions with customers and merchants
df = transactions_df.merge(customers_df, on='customer_id', how='left')
df = df.merge(merchants_df, on='merchant_id', how='left')

# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Combined dataset shape: {df.shape}")
print(f"Target variable: is_fraud")
print(f"Fraud rate: {df['is_fraud'].mean()*100:.2f}%")
print("\n✅ Datasets merged")

Combined dataset shape: (10000, 32)
Target variable: is_fraud
Fraud rate: 6.23%

✅ Datasets merged


In [7]:
#  Feature Engineering - Create New Features
def engineer_features(df):
    """Create new features from existing ones"""
    df_fe = df.copy()
    
    # 1. Time-based features
    df_fe['hour'] = df_fe['timestamp'].dt.hour
    df_fe['day_of_week'] = df_fe['timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday
    df_fe['day_of_month'] = df_fe['timestamp'].dt.day
    df_fe['month'] = df_fe['timestamp'].dt.month
    df_fe['is_weekend'] = df_fe['day_of_week'].isin([5, 6]).astype(int)
    df_fe['is_night'] = ((df_fe['hour'] >= 22) | (df_fe['hour'] <= 5)).astype(int)
    
    # 2. Ratio features
    df_fe['amount_vs_customer_avg'] = df_fe['amount_to_avg_ratio']
    df_fe['amount_vs_merchant_avg'] = df_fe['transaction_amount'] / df_fe['avg_transaction_amount']
    df_fe['amount_vs_merchant_avg'] = df_fe['amount_vs_merchant_avg'].replace([np.inf, -np.inf], 0).fillna(0)
    
    # 3. Customer profile features
    df_fe['customer_age_bracket'] = pd.cut(df_fe['age'], 
                                           bins=[0, 25, 40, 60, 100], 
                                           labels=['Young', 'Adult', 'Middle_Aged', 'Senior'])
    
    df_fe['account_age_years'] = df_fe['account_age_months'] / 12
    df_fe['is_new_customer'] = (df_fe['account_age_months'] < 6).astype(int)
    df_fe['credit_score_bracket'] = pd.cut(df_fe['credit_score'], 
                                           bins=[0, 580, 670, 740, 800, 850], 
                                           labels=['Poor', 'Fair', 'Good', 'Very_Good', 'Excellent'])
    
    # 4. Merchant features
    df_fe['merchant_age_years'] = 2026 - df_fe['established_year']
    df_fe['is_old_merchant'] = (df_fe['merchant_age_years'] > 20).astype(int)
    df_fe['merchant_fraud_rate'] = df_fe['fraud_reports'] / (df_fe['merchant_age_years'] + 1)
    
    # 5. Interaction features
    df_fe['high_risk_customer_high_risk_merchant'] = ((df_fe['risk_category'] == 'High') & 
                                                       df_fe['is_high_risk']).astype(int)
    df_fe['high_amount_high_risk'] = ((df_fe['transaction_amount'] > df_fe['transaction_amount'].median()) & 
                                       df_fe['is_high_risk']).astype(int)
    df_fe['international_online'] = (df_fe['is_international'] & df_fe['is_online']).astype(int)
    
    # 6. Velocity features (requires sorting by timestamp per customer)
    df_fe = df_fe.sort_values(['customer_id', 'timestamp'])
    df_fe['time_since_last_txn_hours'] = df_fe.groupby('customer_id')['timestamp'].diff().dt.total_seconds() / 3600
    df_fe['time_since_last_txn_hours'] = df_fe['time_since_last_txn_hours'].fillna(0)
    
    # 7. Transaction frequency (rolling count per customer)
    df_fe['txn_count_last_24h'] = df_fe.groupby('customer_id')['timestamp'].transform(
        lambda x: x.diff().dt.total_seconds().fillna(0).apply(lambda y: sum(1 for _ in range(1) if y <= 86400))
    )
    # Simplified: use previous_transactions_24h already exists
    
    return df_fe

# Apply feature engineering
df_engineered = engineer_features(df)
print(f"Engineered dataset shape: {df_engineered.shape}")
print(f"New features added: {df_engineered.shape[1] - df.shape[1]}")
print("\n✅ Feature engineering complete")

Engineered dataset shape: (10000, 52)
New features added: 20

✅ Feature engineering complete


In [8]:
# Identify feature types
target = 'is_fraud'
id_cols = ['transaction_id', 'customer_id', 'merchant_id', 'timestamp']

# Numerical features
numerical_features = [
    'transaction_amount', 'age', 'account_age_months', 'credit_score', 
    'account_balance', 'num_credit_cards', 'amount_to_avg_ratio',
    'previous_transactions_24h', 'days_since_last_txn',
    'avg_transaction_amount', 'fraud_reports', 'established_year',
    'hour', 'day_of_week', 'day_of_month', 'month',
    'amount_vs_merchant_avg', 'account_age_years', 'merchant_age_years',
    'merchant_fraud_rate', 'time_since_last_txn_hours'
]

# Ordinal categorical features (ordered)
ordinal_features = [
    'income_bracket',  # Low < Medium < High
    'risk_category'    # Low < Medium < High
]

# Ordinal mappings
income_mapping = [['Low', 'Medium', 'High']]
risk_mapping = [['Low', 'Medium', 'High']]

# Nominal categorical features (one-hot)
nominal_features = [
    'gender', 'employment_status', 'home_ownership', 
    'transaction_type', 'device_type', 'ip_location',
    'merchant_category', 'country',
    'customer_age_bracket', 'credit_score_bracket'
]

# Binary features (already 0/1)
binary_features = [
    'is_online', 'is_international', 'is_high_risk', 
    'is_fraud_history', 'is_weekend', 'is_night',
    'is_new_customer', 'is_old_merchant',
    'high_risk_customer_high_risk_merchant', 'high_amount_high_risk',
    'international_online'
]

# Features to drop (IDs, timestamps, and features that will be encoded)
features_to_drop = id_cols

print(f"Numerical features: {len(numerical_features)}")
print(f"Ordinal features: {len(ordinal_features)}")
print(f"Nominal features: {len(nominal_features)}")
print(f"Binary features: {len(binary_features)}")
print("\n✅ Feature types defined")

Numerical features: 21
Ordinal features: 2
Nominal features: 10
Binary features: 11

✅ Feature types defined


In [9]:
# Prepare Features and Target
# Create feature matrix X and target y
X = df_engineered.drop(columns=features_to_drop + [target])
y = df_engineered[target]

# Store feature names for later
feature_names = X.columns.tolist()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Fraud cases: {y.sum()}")
print(f"Non-fraud cases: {len(y) - y.sum()}")
print("\n✅ Features and target prepared")

X shape: (10000, 47)
y shape: (10000,)
Fraud cases: 623
Non-fraud cases: 9377

✅ Features and target prepared


In [10]:
# Split Data (Time-based)
# Sort by timestamp for time-based split
df_sorted = df_engineered.sort_values('timestamp')
X_sorted = df_sorted.drop(columns=features_to_drop + [target])
y_sorted = df_sorted[target]

# Time-based split: 70% train, 15% val, 15% test
train_size = 0.70
val_size = 0.15
test_size = 0.15

n = len(X_sorted)
train_idx = int(n * train_size)
val_idx = int(n * (train_size + val_size))

X_train = X_sorted.iloc[:train_idx]
X_val = X_sorted.iloc[train_idx:val_idx]
X_test = X_sorted.iloc[val_idx:]

y_train = y_sorted.iloc[:train_idx]
y_val = y_sorted.iloc[train_idx:val_idx]
y_test = y_sorted.iloc[val_idx:]

# Get timestamps for reference
train_timestamps = df_sorted['timestamp'].iloc[:train_idx]
val_timestamps = df_sorted['timestamp'].iloc[train_idx:val_idx]
test_timestamps = df_sorted['timestamp'].iloc[val_idx:]

print(f"Train set: {X_train.shape[0]} ({train_size*100:.0f}%)")
print(f"Validation set: {X_val.shape[0]} ({val_size*100:.0f}%)")
print(f"Test set: {X_test.shape[0]} ({test_size*100:.0f}%)")
print(f"\nTrain fraud rate: {y_train.mean()*100:.2f}%")
print(f"Validation fraud rate: {y_val.mean()*100:.2f}%")
print(f"Test fraud rate: {y_test.mean()*100:.2f}%")
print(f"\nTime range:")
print(f"Train: {train_timestamps.min()} to {train_timestamps.max()}")
print(f"Validation: {val_timestamps.min()} to {val_timestamps.max()}")
print(f"Test: {test_timestamps.min()} to {test_timestamps.max()}")
print("\n✅ Time-based split complete")

Train set: 7000 (70%)
Validation set: 1500 (15%)
Test set: 1500 (15%)

Train fraud rate: 6.10%
Validation fraud rate: 7.13%
Test fraud rate: 5.93%

Time range:
Train: 2026-08-01 00:08:10 to 2026-08-21 23:21:01
Validation: 2026-08-21 23:23:24 to 2026-08-26 12:21:27
Test: 2026-08-26 12:25:54 to 2026-08-30 23:53:55

✅ Time-based split complete


In [11]:
# Create Preprocessing Pipelines
# Preprocessing for numerical features
numerical_transformer = Pipeline(steps=[
    ('scaler', RobustScaler())  # RobustScaler handles outliers better
])

# Preprocessing for ordinal features
ordinal_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=[income_mapping[0], risk_mapping[0]]))
])

# Preprocessing for nominal features
nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine all preprocessors
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('nom', nominal_transformer, nominal_features),
        ('bin', 'passthrough', binary_features)  # binary features pass through
    ],
    remainder='drop'
)

print("Preprocessing pipeline created:")
print(f"  - Numerical: RobustScaler ({len(numerical_features)} features)")
print(f"  - Ordinal: OrdinalEncoder ({len(ordinal_features)} features)")
print(f"  - Nominal: OneHotEncoder ({len(nominal_features)} features)")
print(f"  - Binary: passthrough ({len(binary_features)} features)")
print("\n✅ Preprocessing pipeline defined")

Preprocessing pipeline created:
  - Numerical: RobustScaler (21 features)
  - Ordinal: OrdinalEncoder (2 features)
  - Nominal: OneHotEncoder (10 features)
  - Binary: passthrough (11 features)

✅ Preprocessing pipeline defined


In [12]:
# Feature Selection with SelectKBest
# First, preprocess the training data to get feature importance
print("Preprocessing training data for feature selection...")
X_train_processed = preprocessor.fit_transform(X_train)

print(f"Processed training shape: {X_train_processed.shape}")

# Use SelectKBest with f_classif
k_features = min(50, X_train_processed.shape[1])  # Select top K features
selector = SelectKBest(score_func=f_classif, k=k_features)
selector.fit(X_train_processed, y_train)

# Get feature scores
feature_scores = selector.scores_
print(f"\nTop {k_features} features selected")
print(f"Feature selection completed")
print("\n✅ Feature selector created")

Preprocessing training data for feature selection...
Processed training shape: (7000, 93)

Top 50 features selected
Feature selection completed

✅ Feature selector created


In [13]:
# Create Complete Pipeline with SMOTE
# Create complete pipeline with SMOTE and feature selection
complete_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('selector', selector),
    ('smote', SMOTE(random_state=42, sampling_strategy='auto')),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

print("Complete pipeline with SMOTE:")
print("  1. Preprocessing (scaling + encoding)")
print("  2. Feature selection (SelectKBest)")
print("  3. SMOTE (oversampling) - Applied only to training data")
print("  4. RandomForestClassifier")
print("\n✅ Complete pipeline defined")

Complete pipeline with SMOTE:
  1. Preprocessing (scaling + encoding)
  2. Feature selection (SelectKBest)
  3. SMOTE (oversampling) - Applied only to training data
  4. RandomForestClassifier

✅ Complete pipeline defined


In [14]:
# Save Processed Data and Artifacts
# Create directories for saving
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save splits
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)

y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save timestamps for reference
pd.DataFrame({'timestamp': train_timestamps}).to_csv('../data/processed/train_timestamps.csv', index=False)
pd.DataFrame({'timestamp': val_timestamps}).to_csv('../data/processed/val_timestamps.csv', index=False)
pd.DataFrame({'timestamp': test_timestamps}).to_csv('../data/processed/test_timestamps.csv', index=False)

# Save feature names
feature_info = {
    'all_features': feature_names,
    'numerical_features': numerical_features,
    'ordinal_features': ordinal_features,
    'nominal_features': nominal_features,
    'binary_features': binary_features,
    'target': target,
    'id_cols': id_cols
}
joblib.dump(feature_info, '../models/feature_info.pkl')

# Save preprocessor and selector for inference
joblib.dump(preprocessor, '../models/preprocessor.pkl')
joblib.dump(selector, '../models/selector.pkl')

print("✅ All data splits and artifacts saved:")
print(f"  - X_train: {X_train.shape}")
print(f"  - X_val: {X_val.shape}")
print(f"  - X_test: {X_test.shape}")
print(f"  - Models saved to: ../models/")
print(f"  - Data saved to: ../data/processed/")

✅ All data splits and artifacts saved:
  - X_train: (7000, 47)
  - X_val: (1500, 47)
  - X_test: (1500, 47)
  - Models saved to: ../models/
  - Data saved to: ../data/processed/


In [15]:
# Verify Test Set is Untouched
print("=" * 60)
print("VERIFICATION: TEST SET IS UNTOUCHED")
print("=" * 60)

print("\n1. Test Set Shape:")
print(f"   X_test: {X_test.shape}")
print(f"   y_test: {y_test.shape}")

print("\n2. Test Set Statistics:")
print(f"   Fraud rate: {y_test.mean()*100:.2f}%")
print(f"   Transaction amount mean: ${X_test['transaction_amount'].mean():.2f}")
print(f"   Transaction amount std: ${X_test['transaction_amount'].std():.2f}")

print("\n3. Test Set Time Range:")
print(f"   Start: {test_timestamps.min()}")
print(f"   End: {test_timestamps.max()}")

print("\n4. Verification Checks:")
print(f"   ✓ Test set not used in preprocessing fit")
print(f"   ✓ Test set not used in SMOTE")
print(f"   ✓ Test set not used in feature selection")
print(f"   ✓ Test set not used in model training")

print("\n✅ Test set is isolated and untouched for final evaluation")

VERIFICATION: TEST SET IS UNTOUCHED

1. Test Set Shape:
   X_test: (1500, 47)
   y_test: (1500,)

2. Test Set Statistics:
   Fraud rate: 5.93%
   Transaction amount mean: $1300.77
   Transaction amount std: $4281.52

3. Test Set Time Range:
   Start: 2026-08-26 12:25:54
   End: 2026-08-30 23:53:55

4. Verification Checks:
   ✓ Test set not used in preprocessing fit
   ✓ Test set not used in SMOTE
   ✓ Test set not used in feature selection
   ✓ Test set not used in model training

✅ Test set is isolated and untouched for final evaluation


In [16]:
# Summary of Feature Engineering Pipeline
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print("\n1. New Features Created:")
print(f"   - Time features: hour, day_of_week, day_of_month, month, is_weekend, is_night")
print(f"   - Ratio features: amount_vs_customer_avg, amount_vs_merchant_avg")
print(f"   - Customer features: customer_age_bracket, account_age_years, is_new_customer, credit_score_bracket")
print(f"   - Merchant features: merchant_age_years, is_old_merchant, merchant_fraud_rate")
print(f"   - Interaction features: 3 interaction features")
print(f"   - Velocity features: time_since_last_txn_hours")

print("\n2. Encoding Strategy:")
print(f"   - One-Hot Encoding: {len(nominal_features)} nominal features")
print(f"   - Ordinal Encoding: {len(ordinal_features)} ordinal features")
print(f"   - Pass-through: {len(binary_features)} binary features")

print("\n3. Scaling:")
print(f"   - RobustScaler for {len(numerical_features)} numerical features")

print("\n4. Feature Selection:")
print(f"   - SelectKBest with f_classif")
print(f"   - Top K features: {k_features}")

print("\n5. Class Imbalance:")
print(f"   - SMOTE applied only to training data")
print(f"   - Original fraud rate: {y_train.mean()*100:.2f}%")
print(f"   - After SMOTE: 50% (balanced)")

print("\n6. Data Splits:")
print(f"   - Train: {X_train.shape[0]} ({train_size*100:.0f}%)")
print(f"   - Validation: {X_val.shape[0]} ({val_size*100:.0f}%)")
print(f"   - Test: {X_test.shape[0]} ({test_size*100:.0f}%)")
print(f"   - Split method: Time-based (chronological)")

print("\n7. Artifacts Saved:")
print(f"   - Preprocessor: ../models/preprocessor.pkl")
print(f"   - Selector: ../models/selector.pkl")
print(f"   - Feature info: ../models/feature_info.pkl")
print(f"   - All splits: ../data/processed/")

print("\n" + "=" * 60)
print("✅ FEATURE ENGINEERING COMPLETE")
print("=" * 60)

FEATURE ENGINEERING SUMMARY

1. New Features Created:
   - Time features: hour, day_of_week, day_of_month, month, is_weekend, is_night
   - Ratio features: amount_vs_customer_avg, amount_vs_merchant_avg
   - Customer features: customer_age_bracket, account_age_years, is_new_customer, credit_score_bracket
   - Merchant features: merchant_age_years, is_old_merchant, merchant_fraud_rate
   - Interaction features: 3 interaction features
   - Velocity features: time_since_last_txn_hours

2. Encoding Strategy:
   - One-Hot Encoding: 10 nominal features
   - Ordinal Encoding: 2 ordinal features
   - Pass-through: 11 binary features

3. Scaling:
   - RobustScaler for 21 numerical features

4. Feature Selection:
   - SelectKBest with f_classif
   - Top K features: 50

5. Class Imbalance:
   - SMOTE applied only to training data
   - Original fraud rate: 6.10%
   - After SMOTE: 50% (balanced)

6. Data Splits:
   - Train: 7000 (70%)
   - Validation: 1500 (15%)
   - Test: 1500 (15%)
   - Split met